# Local Model Serving: Ollama vs llama.cpp

A theory-only reference (command examples are shown as reference snippets, not
executable cells) on the two most common ways to run an LLM locally — **Ollama** and
**llama.cpp** — what each actually does under the hood, how the path from "local
experiment" to "deployable service" looks, and when to reach for one over the other.

**Contents**
1. Ollama: the fast path to a local model
2. llama.cpp: direct control over inference
3. llama-server: the OpenAI-compatible API layer
4. Streaming responses
5. Connecting an application (Python client)
6. Fine-grained server control (reasoning, context, threads)
7. Comparison table
8. When to choose Ollama
9. When to choose llama.cpp
10. The deployment path: local → production
11. Extra techniques worth knowing


## 1. Ollama: The Fast Path to a Local Model

**Ollama** is the easiest way to get a local LLM running with almost no setup. It wraps
a llama.cpp-based backend inside a friendly background service.

**Typical workflow (reference commands):**
```powershell
ollama pull qwen3.5:4b     # download a model
ollama run qwen3.5:4b      # interactive terminal chatbot
ollama list                # see installed models
ollama ps                  # check what's currently running
```

**Architecture:**
```text
Your application
       │
       ▼
     Ollama
       │
       ▼
 Qwen3.5 4B
       │
       ▼
    CPU/GPU
```

**What Ollama manages for you:**
- Model downloading and storage
- Model loading/unloading
- Hardware acceleration (CPU/GPU selection)
- API serving (OpenAI-compatible)
- A built-in chat interface
- General model management (switching models, listing, removing)

**Core use case:** *"I want to quickly run and experiment with local LLMs."*

This makes it well suited to a fast iteration loop:
```text
Experimenting → Testing different models → Local chatbot → RAG development → Agent development
```


## 2. llama.cpp: Direct Control Over Inference

**llama.cpp** is the lower-level engine Ollama itself is often built on top of. Using
it directly trades away Ollama's convenience for **direct control over the inference
server** — the natural next step when the eventual goal is deploying the model as a
service rather than just chatting with it locally.

**Verifying installation (reference commands):**
```powershell
llama --version
llama-server --version
```

**Downloading a model straight from Hugging Face:**
```powershell
llama cli -hf lmstudio-community/Qwen3.5-4B-GGUF:Q4_K_M
```
This pulls down the GGUF weights (e.g. `Qwen3.5-4B-Q4_K_M.gguf`) and, for multimodal
models, the vision projector file (`mmproj-Qwen3.5-4B-BF16.gguf`).

**On the `Q4_K_M` naming:** this indicates a **4-bit quantized** GGUF model — a
significant memory reduction compared to full-precision (FP16/BF16) weights, at a
modest cost to output quality. See the "Extra techniques" section below for more on
quantization levels.


## 3. llama-server: The OpenAI-Compatible API Layer

Rather than using `llama cli` for an interactive terminal session, running
**`llama-server`** instead turns the model into an **HTTP API** that any application
can call — this is the piece that actually makes llama.cpp deployable.

**Starting the server (reference command):**
```powershell
llama-server `
  -hf lmstudio-community/Qwen3.5-4B-GGUF:Q4_K_M `
  --reasoning off `
  --ctx-size 4096 `
  --threads 8 `
  --port 8080
```
This exposes an HTTP endpoint at `http://localhost:8080`.

**Architecture:**
```text
                HTTP
Your Python App ──────────► llama-server
                              │
                              ▼
                       Qwen3.5 4B GGUF
```

Instead of an application loading the GGUF weights directly, it talks to the server
over HTTP — the same shape as talking to any hosted LLM API.

**Verifying the server is up (reference command):**
```powershell
curl.exe http://localhost:8080/v1/models
```
A successful response confirms the model loaded and is exposed through the API,
following the **OpenAI-compatible API** spec — the same request/response shape used by
OpenAI's own endpoints, which is what makes it a drop-in replacement for many existing
client libraries.


## 4. Streaming Responses

For a chatbot-style UX, waiting for the entire response before showing anything feels
slow. Enabling streaming changes the experience from:

```text
User → wait several seconds → complete answer
```
to:
```text
User → generation starts → chunk → chunk → chunk → chunk → complete answer
```

**Enabling it (reference):** set `"stream": true` in the request body.

```powershell
curl.exe -N `
    -H "Content-Type: application/json" `
    --data-raw "$body" `
    http://localhost:8080/v1/chat/completions
```
The `-N` flag disables curl's output buffering so chunks are printed as they arrive,
rather than all at once at the end.


## 5. Connecting an Application (Python Client)

Because `llama-server` speaks the OpenAI-compatible API, it can be used with the
standard **OpenAI Python client** just by pointing it at a local URL instead of
OpenAI's servers.

**Architecture:**
```text
Python
  │  OpenAI-compatible API
  ▼
localhost:8080
  │
  ▼
llama-server
  │
  ▼
Qwen3.5 4B
```

**Client setup (reference):**
```python
from openai import OpenAI

client = OpenAI(
    base_url="http://localhost:8080/v1",
    api_key="not-needed",
)
```

**Streaming a response (reference):**
```python
stream = client.chat.completions.create(
    model="lmstudio-community/Qwen3.5-4B-GGUF:Q4_K_M",
    messages=messages,
    temperature=0,
    max_tokens=500,
    stream=True,
)

for chunk in stream:
    if chunk.choices and chunk.choices[0].delta.content:
        print(chunk.choices[0].delta.content, end="", flush=True)
```
This is the same client code pattern used for a hosted OpenAI model — only the
`base_url` changes — which is what makes swapping between local and hosted inference
relatively painless during development.


## 6. Fine-Grained Server Control

One of the practical advantages of `llama-server` over Ollama is **server-level
control** of behavior that would otherwise need to be handled per-request in
application code.

**Reasoning control:** `llama-server` supports a `--reasoning` flag with three modes:
```text
--reasoning on     # always produce reasoning/thinking output
--reasoning off     # skip straight to the final answer
--reasoning auto    # let the model decide
```
Setting this **at the server level** (e.g. `--reasoning off`) means the application
doesn't need to pass a per-request parameter like `reasoning=False` — the behavior is
fixed centrally for every request the server handles.

**Other server-level parameters** worth knowing:
- `--ctx-size` — the context window size available to the model.
- `--threads` — CPU thread count used for inference.
- `--port` — which port the HTTP API is exposed on.

This kind of centralized, server-side configuration is exactly the "more granular
control" that motivates moving from Ollama to llama.cpp in the first place.


## 7. Comparison Table

| | Ollama | llama.cpp |
|---|---|---|
| Main purpose | Easy local LLM usage | Low-level/local LLM inference |
| Setup | Very easy | More manual |
| Model management | Excellent (built-in) | More manual |
| GGUF support | Yes (abstracted) | Native |
| Local experimentation | ⭐⭐⭐⭐⭐ | ⭐⭐⭐⭐ |
| API | Yes | Yes |
| OpenAI-compatible API | Yes | Yes |
| Server control | Less granular | More granular |
| Quantization control | Mostly abstracted | Direct |
| CPU optimization | Good | Excellent / control-oriented |
| Deployment control | Good | **Better** |
| Production customization | Moderate | **High** |
| Best for | Development / experimentation | Custom inference / deployment |


## 8. When to Choose Ollama

Choose Ollama when the goal is simply: *"I just want to run a model locally."*

Good fits include:
- Learning about local LLMs for the first time
- Testing and comparing different models quickly
- RAG experiments
- Early-stage agent development
- A quick local chatbot
- Frequently switching between models with minimal friction

This requires very little infrastructure work — a single `ollama run <model>` command
gets you a working chatbot.


## 9. When to Choose llama.cpp

Choose llama.cpp when the goal is: *"I want control over how inference works, and I
eventually want to deploy this."*

Good fits include:
- Building a custom LLM server rather than using a pre-packaged one
- CPU-focused inference where you want direct optimization control
- Working with GGUF models and specific quantization levels directly
- Needing a streaming, OpenAI-compatible API for an application
- Custom context-size or thread configuration
- Explicit reasoning on/off/auto control
- Production-oriented deployment
- Running the model behind your own API rather than Ollama's abstraction layer


## 10. The Deployment Path: Local → Production

A typical progression looks like this. It starts fully local:

```text
                   LOCAL
                     │
                     ▼
              llama-server
                     │
             ┌───────┴───────┐
             │               │
          REST API       Web UI
             │
             ▼
        Python Chatbot
             │
             ▼
       Qwen3.5 4B GGUF
```

...and evolves toward an internet-facing service by adding an authentication/HTTPS
layer and an application server in front of `llama-server`:

```text
                  INTERNET
                     │
                     ▼
              HTTPS / Auth
                     │
                     ▼
             Your API / FastAPI
                     │
                     ▼
              llama-server
                     │
                     ▼
             Qwen3.5 4B GGUF
```

**In short:** Ollama is the quick local-experimentation path, while llama.cpp +
`llama-server` is the better foundation for building and eventually deploying your own
LLM inference service.


## 11. Extra Techniques Worth Knowing

A few additional capabilities on each side that extend the comparison above.

### Ollama extras
- **Modelfiles** — a Dockerfile-like config that lets you customize a base model's
  system prompt, default parameters (temperature, context size), and even create
  named custom model variants (`ollama create`).
- **`OLLAMA_HOST`** — environment variable to bind Ollama's API to a network interface
  other than localhost, useful for accessing it from another machine on the same
  network during development.
- **`OLLAMA_MODELS`** — environment variable to control where model weights are
  stored on disk, useful when the default drive doesn't have enough space.

### llama.cpp extras
- **`--n-gpu-layers`** — controls how many model layers get offloaded to GPU vs. kept
  on CPU, letting you tune the CPU/GPU split for hardware that can't fit the whole
  model in VRAM.
- **`--parallel`** — enables handling multiple concurrent requests/sequences on a
  single server instance, a step toward the kind of concurrent serving that dedicated
  production engines (like vLLM) specialize in.
- **`--grammar`** — constrains output to a formal grammar (e.g. forcing valid JSON),
  useful for structured-output use cases.
- **Multimodal (`mmproj`) support** — a separate vision-projector GGUF file lets a
  compatible model accept image inputs alongside text.

### Where this path leads next
Both Ollama and llama.cpp are fundamentally **single-machine, low-concurrency**
tools — excellent for development and lightweight production, but not designed for
serving hundreds of simultaneous users efficiently. When traffic grows past that
point, the natural next step is a dedicated high-throughput serving engine such as
**vLLM** or **SGLang**, which handle request batching and GPU memory management at a
level neither Ollama nor plain llama.cpp are built for.
